# Юдин Артём М4121

In [1]:
import pandas as pd
from sklearn.ensemble import (
    AdaBoostClassifier,
    BaggingClassifier,
    RandomForestClassifier,
    StackingClassifier,
    VotingClassifier,
)
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_recall_fscore_support,
    precision_score,
    recall_score,
)
from sklearn.model_selection import GridSearchCV
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier

In [2]:
X_train = pd.read_csv("../data/preprocessed/X_train_smart_regex_7020samples.csv")
X_val = pd.read_csv("../data/preprocessed/X_val_smart_regex_1980samples.csv")

y_train = pd.read_csv("../data/preprocessed/y_train_7020samples.csv")
y_val = pd.read_csv("../data/preprocessed/y_val_1980samples.csv")

# объединение токенов вместе
for df in (X_train, X_val):
    df["untokenized_ttext"] = df["ttext"].apply(lambda x: " ".join(eval(x)))

df = 5e-4
# инициализация векторизаторов
tdidf_vectorizer = TfidfVectorizer(min_df=df, max_df=1 - df)

# инициализируем скейлеры
tfidf_scaler = StandardScaler()

le = LabelEncoder()
y_train = le.fit_transform(y_train)
y_val = le.transform(y_val)

# кодируем и масштабируем данные
X_train_tfidf = tfidf_scaler.fit_transform(
    tdidf_vectorizer.fit_transform(X_train["untokenized_ttext"]).toarray()
)
X_val_tfidf = tfidf_scaler.transform(
    tdidf_vectorizer.transform(X_val["untokenized_ttext"]).toarray()
)

/home/artyom/myprojects/ITMO/IIML/.venv/lib/python3.12/site-packages/sklearn/preprocessing/_label.py:110: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
/home/artyom/myprojects/ITMO/IIML/.venv/lib/python3.12/site-packages/sklearn/preprocessing/_label.py:129: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, dtype=self.classes_.dtype, warn=True)


# Подбор гиперпараметров

In [ ]:
param_grid = {
    "n_estimators": list(range(10, 150, 45)),
    "criterion": ["gini", "entropy"],
    "max_depth": list(range(10, 30, 5)),
    "min_samples_leaf": list(range(2, 8, 4)),
}
gs = GridSearchCV(
    RandomForestClassifier(random_state=42),
    param_grid=param_grid,
    cv=5,
    scoring="accuracy",
    n_jobs=-1,
)
gs.fit(X_train_tfidf, y_train)

,estimator,RandomForestC...ndom_state=42)
,param_grid,"{'criterion': ['gini', 'entropy'], 'max_depth': [10, 15, ...], 'min_samples_leaf': [2, 6], 'n_estimators': [10, 55, ...]}"
,scoring,'accuracy'
,n_jobs,-1
,refit,True
,cv,5
,verbose,0
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,n_estimators,100


In [ ]:
gs.best_params_

{'criterion': 'entropy',
 'max_depth': 25,
 'min_samples_leaf': 6,
 'n_estimators': 100}

## Получение новых метрик

In [ ]:
y_pred = gs.predict(X_val_tfidf)
(
    accuracy_score(y_val, y_pred),
    precision_recall_fscore_support(y_val, y_pred, labels=[1]),
)

(0.6505050505050505,
 (array([0.64272031]), array([0.67777778]), array([0.65978368]), array([990])))

# Обучение ансамблей

In [3]:
def train(X_train_tfidf, y_train, X_val_tfidf, y_val) -> dict[str, list]:
    metrics = {
        "Accuracy": accuracy_score,
        "Precision": precision_score,
        "Recall": recall_score,
        "F1": f1_score,
    }

    models = {
        "KNN": KNeighborsClassifier(weights="distance"),
        "GaussianNB": GaussianNB(),
        "Voting": VotingClassifier(
            [("nb", GaussianNB()), ("knn", KNeighborsClassifier(weights="distance"))],
            voting="soft",
        ),
        "Stacking": StackingClassifier(
            [("nb", GaussianNB()), ("knn", KNeighborsClassifier(weights="distance"))]
        ),
        "Decision Tree": DecisionTreeClassifier(random_state=42),
        "Bagging": BaggingClassifier(
            DecisionTreeClassifier(random_state=42), n_estimators=10
        ),
        "Boosting": AdaBoostClassifier(DecisionTreeClassifier(random_state=42)),
    }
    df_dict = {"Model": [], "Accuracy": [], "Precision": [], "Recall": [], "F1": []}

    for model_name, model in models.items():
        df_dict["Model"].append(model_name)

        # обучаем модель
        model.fit(X_train_tfidf, y_train)
        y_pred = model.predict(X_val_tfidf)

        # рассчитываем метрики для обученного набора
        for metric_name, metric in metrics.items():
            df_dict[metric_name].append(round(metric(y_val, y_pred), 2))

    return df_dict

In [4]:
df = pd.DataFrame(train(X_train_tfidf, y_train, X_val_tfidf, y_val))
df

,Model,Accuracy,Precision,Recall,F1
0,KNN,0.59,0.62,0.48,0.54
1,GaussianNB,0.59,0.65,0.41,0.50
2,Voting,0.59,0.65,0.41,0.50
3,Stacking,0.61,0.64,0.49,0.55
4,Decision Tree,0.62,0.63,0.57,0.60
5,Bagging,0.64,0.66,0.57,0.61
6,Boosting,0.62,0.63,0.60,0.61


# Выводы
- Подбор гиперпараметров улучшает метрики на выборке, на которой подбор происходит, но вместе с тем не гарантирует прироста на других наборах данных
- Методы ансамблирования могут быть использованы даже для улучшения качества работы алгоритмов, которые выдают не самые лучшеи показатели
- Разные методы ансамблирования могут понижать количество как ложноположительных, так и ложноотрицательных предсказаний у моделей